# Consultas

## Conexión

In [ ]:
import os
import re
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

load_dotenv()   # la contraseña está en el .env como PASS_SQL
PASS = os.getenv("PASS_SQL")

In [ ]:
def url(bd):
    return URL.create("postgresql+psycopg2", username="postgres", password=PASS,
                      host="localhost", port=5432, database=bd)

# Creamos la base de datos si no existe (hay que conectarse a la de por defecto, "postgres")
admin = create_engine(url("postgres"), isolation_level="AUTOCOMMIT")
with admin.connect() as conn:
    existe = conn.execute(text("SELECT 1 FROM pg_database WHERE datname = 'pipeline_deudas'")).scalar()
    if not existe:
        conn.execute(text("CREATE DATABASE pipeline_deudas"))

engine = create_engine(url("pipeline_deudas"))

In [ ]:
# Atajo para lanzar consultas y que salgan como DataFrame
def q(sql):
    return pd.read_sql(text(sql), engine)

q("SELECT version()")

## Tablas

In [ ]:
# Sin clave foránea entre expedientes y clientes porque hay expedientes con un cliente que no está en clientes
ddl = [
    "DROP TABLE IF EXISTS expedientes",
    "DROP TABLE IF EXISTS clientes",
    "DROP TABLE IF EXISTS oficinas",
    """
    CREATE TABLE oficinas (
        nombre_oficina          TEXT PRIMARY KEY,
        responsable             TEXT,
        objetivo_conversion_pct NUMERIC(4,1),
        fecha_apertura_oficina  DATE
    )""",
    """
    CREATE TABLE clientes (
        cliente_id          INT PRIMARY KEY,
        nombre_completo     TEXT,
        fecha_entrada       DATE,
        canal_captacion     TEXT,
        departamento_oficina TEXT,
        codigo_postal       CHAR(5),
        email               TEXT,
        telefono            CHAR(9),
        ingresos_mensuales  NUMERIC(10,2),
        tiene_aval          BOOLEAN
    )""",
    """
    CREATE TABLE expedientes (
        expediente_id           INT PRIMARY KEY,
        cliente_id              INT,
        fecha_apertura          DATE,
        deuda_total             NUMERIC(12,2),
        num_acreedores          INT,
        acreedores              TEXT,
        cuota_mensual_propuesta NUMERIC(10,2),
        estado_expediente       TEXT
    )""",
]

with engine.begin() as conn:
    for s in ddl:
        conn.execute(text(s))

## Carga de los CSV

In [ ]:
CLEAN = Path("..") / "clean"

def a_snake(col):
    # ClienteId -> cliente_id
    return re.sub(r"(?<!^)(?=[A-Z])", "_", col).lower()

def cargar(archivo, tabla, **kwargs):
    df = pd.read_csv(CLEAN / archivo, sep=";", **kwargs)
    df.columns = [a_snake(c) for c in df.columns]
    df.to_sql(tabla, engine, if_exists="append", index=False)
    return len(df)

# Teléfono y CP como texto para no perder los ceros. Si falla por clave duplicada, mirar los duplicados en el EDA
print(cargar("oficinas_clean.csv", "oficinas", parse_dates=["FechaAperturaOficina"]))
print(cargar("clientes_clean.csv", "clientes",
             dtype={"CodigoPostal": str, "Telefono": str, "TieneAval": "boolean"},
             parse_dates=["FechaEntrada"]))
print(cargar("expedientes_clean.csv", "expedientes",
             dtype={"ClienteId": "Int64", "NumAcreedores": "Int64"},
             parse_dates=["FechaApertura"]))

## Consultas

Primero un vistazo

In [ ]:
q("""
SELECT 'clientes' AS tabla, COUNT(*) AS filas FROM clientes
UNION ALL SELECT 'expedientes', COUNT(*) FROM expedientes
UNION ALL SELECT 'oficinas', COUNT(*) FROM oficinas
""")

In [ ]:
q("SELECT * FROM clientes LIMIT 5")

Expedientes por estado

In [ ]:
q("""
SELECT estado_expediente,
       COUNT(*) AS expedientes,
       ROUND(AVG(deuda_total), 0) AS deuda_media
FROM expedientes
GROUP BY estado_expediente
ORDER BY expedientes DESC
""")

Deuda por oficina (join con clientes)

In [ ]:
q("""
SELECT c.departamento_oficina,
       COUNT(*) AS expedientes,
       SUM(e.deuda_total) AS deuda_total
FROM expedientes e
JOIN clientes c ON c.cliente_id = e.cliente_id
GROUP BY c.departamento_oficina
ORDER BY deuda_total DESC
""")

Lo mismo pero con el responsable de la oficina (3 tablas)

In [ ]:
q("""
SELECT o.nombre_oficina, o.responsable, o.objetivo_conversion_pct,
       COUNT(DISTINCT c.cliente_id) AS clientes,
       COUNT(e.expediente_id) AS expedientes,
       SUM(e.deuda_total) AS deuda_total
FROM oficinas o
LEFT JOIN clientes c ON c.departamento_oficina = o.nombre_oficina
LEFT JOIN expedientes e ON e.cliente_id = c.cliente_id
GROUP BY o.nombre_oficina, o.responsable, o.objetivo_conversion_pct
ORDER BY deuda_total DESC
""")

Canal de captación, deuda media

In [ ]:
q("""
SELECT c.canal_captacion,
       COUNT(*) AS expedientes,
       ROUND(AVG(e.deuda_total), 0) AS deuda_media
FROM expedientes e
JOIN clientes c ON c.cliente_id = e.cliente_id
GROUP BY c.canal_captacion
ORDER BY deuda_media DESC
""")

Top 3 expedientes con más deuda de cada oficina (función de ventana)

In [ ]:
q("""
SELECT *
FROM (
    SELECT c.departamento_oficina, e.expediente_id, e.deuda_total,
           ROW_NUMBER() OVER (PARTITION BY c.departamento_oficina ORDER BY e.deuda_total DESC) AS puesto
    FROM expedientes e
    JOIN clientes c ON c.cliente_id = e.cliente_id
    WHERE e.deuda_total IS NOT NULL
) t
WHERE puesto <= 3
ORDER BY departamento_oficina, puesto
""")

Expedientes por mes, acumulado y diferencia con el mes anterior

In [ ]:
q("""
WITH mensual AS (
    SELECT DATE_TRUNC('month', fecha_apertura)::date AS mes, COUNT(*) AS n
    FROM expedientes
    WHERE fecha_apertura IS NOT NULL
    GROUP BY 1
)
SELECT mes, n,
       SUM(n) OVER (ORDER BY mes) AS acumulado,
       n - LAG(n) OVER (ORDER BY mes) AS dif_mes_anterior
FROM mensual
ORDER BY mes
""")

% de incumplidos por oficina

In [ ]:
q("""
SELECT c.departamento_oficina,
       COUNT(*) AS expedientes,
       SUM(CASE WHEN e.estado_expediente = 'Incumplido' THEN 1 ELSE 0 END) AS incumplidos,
       ROUND(100.0 * SUM(CASE WHEN e.estado_expediente = 'Incumplido' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_incumplidos
FROM expedientes e
JOIN clientes c ON c.cliente_id = e.cliente_id
GROUP BY c.departamento_oficina
ORDER BY pct_incumplidos DESC
""")

Cuota propuesta respecto a los ingresos (los que se llevan más de la mitad)

In [ ]:
q("""
SELECT e.expediente_id, c.cliente_id, c.ingresos_mensuales, e.cuota_mensual_propuesta,
       ROUND(100.0 * e.cuota_mensual_propuesta / c.ingresos_mensuales, 1) AS pct_cuota
FROM expedientes e
JOIN clientes c ON c.cliente_id = e.cliente_id
WHERE c.ingresos_mensuales > 0
  AND e.cuota_mensual_propuesta > 0.5 * c.ingresos_mensuales
ORDER BY pct_cuota DESC
LIMIT 10
""")

Clientes sin ningún expediente

In [ ]:
q("""
SELECT c.cliente_id, c.nombre_completo, c.departamento_oficina
FROM clientes c
LEFT JOIN expedientes e ON e.cliente_id = c.cliente_id
WHERE e.expediente_id IS NULL
""")

Expedientes con un cliente que no existe en clientes

In [ ]:
q("""
SELECT e.expediente_id, e.cliente_id, e.deuda_total
FROM expedientes e
LEFT JOIN clientes c ON c.cliente_id = e.cliente_id
WHERE e.cliente_id IS NOT NULL
  AND c.cliente_id IS NULL
""")

Expedientes sin cliente asignado

In [ ]:
q("SELECT COUNT(*) AS sin_cliente FROM expedientes WHERE cliente_id IS NULL")